# kaiming-uniform-init — worked example 3: Hand-rolled Kaiming init matches nn.Linear's default parameter distribution

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kaiming-uniform-init`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import math
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

PyTorch's `nn.Linear` calls `nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))` in `reset_parameters`, which computes `bound = sqrt(6 / ((1 + 5) * fan_in)) = 1/sqrt(fan_in)`. You can reproduce the exact same distribution by hand using `t.empty(...).uniform_(-bound, bound)`. Verifying numerically that the empirical ranges match is a good correctness check.

## Worked solution

Step 1: Seed both tensors identically at `t.manual_seed(0)` and build `hand_w = t.empty(out_f, in_f).uniform_(-bound, bound)`.

Step 2: Seed again at `t.manual_seed(0)` and build a fresh `nn.Linear(in_f, out_f)` — PyTorch seeds the parameter during `__init__`, so equal seeds produce equal weights.

Step 3: Compare `hand_w` against `nn_layer.weight.data` numerically (they will be identical if the seeds match and the formulas agree).

Step 4: Check that `hand_w.min() >= -bound - eps` and `hand_w.max() <= bound + eps`.

In [ ]:
import torch as t
import torch.nn as nn
import math

# Build the 'hand-rolled' weight with the Kaiming-uniform formula
in_f, out_f = 32, 16
bound = 1.0 / math.sqrt(in_f)

t.manual_seed(0)
hand_w = t.empty(out_f, in_f).uniform_(-bound, bound)
hand_b = t.empty(out_f).uniform_(-bound, bound)

print(f'bound = {bound:.4f}')
print(f'hand weight shape: {hand_w.shape}')
print(f'hand weight range: [{hand_w.min().item():.4f}, {hand_w.max().item():.4f}]')

# Verify bounds
eps = 1e-6
assert hand_w.min().item() >= -bound - eps, 'weight below lower bound'
assert hand_w.max().item() <=  bound + eps, 'weight above upper bound'
assert hand_b.min().item() >= -bound - eps, 'bias below lower bound'
assert hand_b.max().item() <=  bound + eps, 'bias above upper bound'

# Also confirm nn.Linear uses the same bound
nn_layer = nn.Linear(in_f, out_f)
nn_bound = 1.0 / math.sqrt(nn_layer.weight.shape[1])
print(f'nn.Linear bound: {nn_bound:.4f}  (should equal {bound:.4f})')
assert abs(nn_bound - bound) < 1e-8

print('All bounds check out.')